# HAM10000 Skin-Lesion Classification — Deep Learning (PyTorch)

**Dataset:** HAM10000 via Kaggle API  
**Model:** EfficientNet-B0 transfer learning  
**Runtime:** Google Colab, T4 GPU when available  
**Task:** 7-class dermoscopic skin-lesion classification

This notebook uses **deep learning only** for the image classifier. It includes lesion-level splitting, class-weighted loss, augmentation, checkpointing, Macro-F1 model selection, and held-out test evaluation.

> Research/educational use only; not a clinical diagnostic system.


In [ ]:
# 1. Install dependencies
!pip -q install kaggle scikit-learn pandas matplotlib seaborn pillow tqdm


In [ ]:
# 2. Imports, seed, and GPU check
import os, glob, zipfile, copy, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: T4/GPU is not enabled. In Colab use Runtime > Change runtime type > T4 GPU.")


In [ ]:
# 3. Kaggle API authentication
# Upload kaggle.json from Kaggle > Settings > API > Create New Token
from google.colab import files

uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured.")


In [ ]:
# 4. Download and extract HAM10000 using Kaggle API
DATA_ROOT = "/content/ham10000"
os.makedirs(DATA_ROOT, exist_ok=True)

KAGGLE_DATASET = "kmader/skin-cancer-mnist-ham10000"

!kaggle datasets download -d {KAGGLE_DATASET} -p /content/ham10000 --force

for z in glob.glob(DATA_ROOT + "/*.zip"):
    print("Extracting:", z)
    with zipfile.ZipFile(z, "r") as f:
        f.extractall(DATA_ROOT)

print("Extraction complete.")


In [ ]:
# 5. Load metadata and map image paths
metadata_candidates = glob.glob(
    DATA_ROOT + "/**/HAM10000_metadata.csv",
    recursive=True
)
if not metadata_candidates:
    raise FileNotFoundError("HAM10000_metadata.csv not found.")

METADATA_PATH = metadata_candidates[0]
df = pd.read_csv(METADATA_PATH)

image_files = glob.glob(DATA_ROOT + "/**/*.jpg", recursive=True)
image_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_files}

df["image_path"] = df["image_id"].map(image_map)
df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Images:", len(df))
print("Unique lesions:", df["lesion_id"].nunique())
print(df.head())


In [ ]:
# 6. Class distribution
class_names = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
print(df["dx"].value_counts().reindex(class_names, fill_value=0))

plt.figure(figsize=(9,4))
sns.countplot(data=df, x="dx", order=class_names)
plt.title("HAM10000 Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Images")
plt.tight_layout()
plt.show()


In [ ]:
# 7. Lesion-level stratified split
# No lesion_id is shared between train/validation/test.

sgkf = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=SEED
)

folds = list(sgkf.split(df.index, df["dx"], df["lesion_id"]))

test_idx = folds[0][1]
val_idx = folds[1][1]
train_idx = sorted(set(df.index) - set(test_idx) - set(val_idx))

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} images / {train_df.lesion_id.nunique()} lesions")
print(f"Val:   {len(val_df)} images / {val_df.lesion_id.nunique()} lesions")
print(f"Test:  {len(test_df)} images / {test_df.lesion_id.nunique()} lesions")

assert set(train_df.lesion_id).isdisjoint(val_df.lesion_id)
assert set(train_df.lesion_id).isdisjoint(test_df.lesion_id)
assert set(val_df.lesion_id).isdisjoint(test_df.lesion_id)
print("Leakage check: PASSED")


## 8. Image preprocessing and augmentation

EfficientNet-B0 uses 224×224 inputs. Training receives random augmentation; validation and test remain deterministic.

The ImageNet normalization is appropriate because the backbone is initialized from ImageNet-pretrained weights.


In [ ]:
# 9. Transforms
IMAGE_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


In [ ]:
# 10. Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = {c:i for i,c in enumerate(class_names)}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = self.class_to_idx[row["dx"]]
        return image, label

train_dataset = HAM10000Dataset(train_df, train_transform)
val_dataset = HAM10000Dataset(val_df, eval_transform)
test_dataset = HAM10000Dataset(test_df, eval_transform)

print(len(train_dataset), len(val_dataset), len(test_dataset))


In [ ]:
# 11. DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))
print("Image batch:", images.shape)
print("Label batch:", labels.shape)


In [ ]:
# 12. Class-weighted loss
train_counts = train_df["dx"].value_counts().reindex(class_names)
class_weights_np = len(train_df) / (len(class_names) * train_counts.values)

class_weights = torch.tensor(
    class_weights_np, dtype=torch.float32, device=device
)

print(train_counts)
print("\nClass weights:")
for c, w in zip(class_names, class_weights_np):
    print(f"{c}: {w:.4f}")


In [ ]:
# 13. Build EfficientNet-B0
try:
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
except Exception as e:
    print("Pretrained weights unavailable:", e)
    model = models.efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))
model = model.to(device)

print(model.classifier)


In [ ]:
# 14. Loss, optimizer, scheduler, mixed precision
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

amp_enabled = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("AMP enabled:", amp_enabled)


In [ ]:
# 15. Train/evaluate functions
def metrics_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    ys, ps = [], []

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=amp_enabled
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        ys.extend(labels.detach().cpu().numpy())
        ps.extend(outputs.argmax(1).detach().cpu().numpy())

    return total_loss / len(loader.dataset), metrics_dict(ys, ps)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    ys, ps, probs = [], [], []

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)
        p = torch.softmax(outputs, dim=1)

        total_loss += loss.item() * images.size(0)
        ys.extend(labels.cpu().numpy())
        ps.extend(outputs.argmax(1).cpu().numpy())
        probs.extend(p.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        metrics_dict(ys, ps),
        np.array(ys),
        np.array(ps),
        np.array(probs)
    )


In [ ]:
# 16. Train with validation Macro F1 checkpointing
EPOCHS = 15
PATIENCE = 5

CHECKPOINT_DIR = "/content/ham10000_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_f1 = -1
best_epoch = 0
best_state = None
no_improve = 0
history = []

start_total = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_m = train_one_epoch(model, train_loader)
    val_loss, val_m, _, _, _ = evaluate(model, val_loader)

    scheduler.step(val_m["macro_f1"])
    lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_m["accuracy"],
        "train_macro_f1": train_m["macro_f1"],
        "val_accuracy": val_m["accuracy"],
        "val_balanced_accuracy": val_m["balanced_accuracy"],
        "val_macro_precision": val_m["macro_precision"],
        "val_macro_recall": val_m["macro_recall"],
        "val_macro_f1": val_m["macro_f1"],
        "lr": lr,
        "epoch_time_min": (time.time()-t0)/60
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train F1 {train_m['macro_f1']:.4f} | "
        f"Val F1 {val_m['macro_f1']:.4f} | "
        f"Val Bal Acc {val_m['balanced_accuracy']:.4f} | "
        f"LR {lr:.2e}"
    )

    if val_m["macro_f1"] > best_val_f1:
        best_val_f1 = val_m["macro_f1"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        torch.save(
            {
                "model_state_dict": best_state,
                "class_names": class_names,
                "best_val_macro_f1": best_val_f1,
                "epoch": best_epoch
            },
            os.path.join(CHECKPOINT_DIR, "efficientnet_b0_best.pth")
        )
        no_improve = 0
        print("  -> Best checkpoint saved.")
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print("Early stopping.")
        break

print(f"Total training time: {(time.time()-start_total)/60:.2f} min")
print(f"Best validation Macro F1: {best_val_f1:.4f} at epoch {best_epoch}")


In [ ]:
# 17. Training curves
history_df = pd.DataFrame(history)

plt.figure(figsize=(8,5))
plt.plot(history_df.epoch, history_df.train_loss, marker="o", label="Train Loss")
plt.plot(history_df.epoch, history_df.val_loss, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history_df.epoch, history_df.train_macro_f1, marker="o", label="Train Macro F1")
plt.plot(history_df.epoch, history_df.val_macro_f1, marker="o", label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Macro F1 Curves")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# 18. Load best checkpoint and evaluate held-out test set
checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "efficientnet_b0_best.pth"
)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_m, y_true, y_pred, y_prob = evaluate(
    model, test_loader
)

print("FINAL TEST RESULTS")
print("------------------")
for k, v in test_m.items():
    print(f"{k}: {v:.4f}")


In [ ]:
# 19. Per-class metrics
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
))


In [ ]:
# 20. Confusion matrices
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EfficientNet-B0 — Test Confusion Matrix")
plt.tight_layout()
plt.show()

cm_norm = confusion_matrix(
    y_true, y_pred, normalize="true"
)

plt.figure(figsize=(8,7))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EfficientNet-B0 — Normalized Test Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# 21. Multiclass ROC-AUC (one-vs-rest)
try:
    y_true_bin = label_binarize(
        y_true, classes=np.arange(len(class_names))
    )
    macro_auc = roc_auc_score(
        y_true_bin, y_prob,
        average="macro", multi_class="ovr"
    )
    weighted_auc = roc_auc_score(
        y_true_bin, y_prob,
        average="weighted", multi_class="ovr"
    )
    print(f"Macro ROC-AUC: {macro_auc:.4f}")
    print(f"Weighted ROC-AUC: {weighted_auc:.4f}")
except ValueError as e:
    macro_auc = np.nan
    weighted_auc = np.nan
    print("ROC-AUC unavailable:", e)


In [ ]:
# 22. Save metrics, history, and checkpoint metadata
RESULT_DIR = "/content/ham10000_deep_learning_results"
os.makedirs(RESULT_DIR, exist_ok=True)

test_results = pd.DataFrame([{
    "Model": "EfficientNet-B0",
    "Accuracy": test_m["accuracy"],
    "Balanced Accuracy": test_m["balanced_accuracy"],
    "Macro Precision": test_m["macro_precision"],
    "Macro Recall": test_m["macro_recall"],
    "Macro F1": test_m["macro_f1"],
    "Macro ROC-AUC": macro_auc,
    "Weighted ROC-AUC": weighted_auc,
    "Best Validation Macro F1": best_val_f1,
    "Best Epoch": best_epoch
}])

test_results.to_csv(
    os.path.join(RESULT_DIR, "efficientnet_b0_test_results.csv"),
    index=False
)
history_df.to_csv(
    os.path.join(RESULT_DIR, "training_history.csv"),
    index=False
)

with open(
    os.path.join(RESULT_DIR, "classification_report.txt"), "w"
) as f:
    f.write(classification_report(
        y_true, y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    ))

print("Saved:", RESULT_DIR)


## 23. Optional Google Drive backup

The dataset is downloaded through Kaggle. Drive is only used for persistent checkpoints/results.

```python
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Skin_Cancer_Project'
!mkdir -p "$DRIVE_DIR"
!cp -r /content/ham10000_checkpoints "$DRIVE_DIR/"
!cp -r /content/ham10000_deep_learning_results "$DRIVE_DIR/"
```


## Research interpretation

Report accuracy together with balanced accuracy, macro precision/recall/F1, per-class performance, confusion matrix, and ROC-AUC where valid. HAM10000 is class-imbalanced, so accuracy alone is insufficient.

The lesion-level split is important because multiple images can belong to the same lesion. Do not describe this model as clinically validated based on HAM10000 performance alone.
